### Config

In [59]:
import ee
import geemap

try:
  import rasterio
except:
  !pip install rasterio
  import rasterio

try:
  import geopandas as gpd
except:
  !pip install geopandas
  import geopandas as gpd

try:
  import shapely
except:
  !pip install shapely
  import shapely

In [60]:
ee.Initialize(project='geemap-496017')

Map = geemap.Map()
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [61]:
#Config
startDate = '2020-01-01'
endDate = '2021-01-01'
site_name = 'Basilicata'
year = '2020'

In [62]:
import rasterio
import geopandas as gpd
from shapely.geometry import box

# Input raster
raster_path = "input/LE_D26_basilicata_2020_data.tif"

# Read raster extent
with rasterio.open(raster_path) as src:
    bounds = src.bounds          # left, bottom, right, top
    crs = src.crs

# Create AOI polygon from raster extent
aoi = gpd.GeoDataFrame(
    geometry=[box(bounds.left, bounds.bottom, bounds.right, bounds.top)],
)

# Assign the raster CRS manually
aoi = aoi.set_crs("EPSG:6875")   # Replace with the correct EPSG

# Reproject to WGS84 for display in geemap
aoi = aoi.to_crs(epsg=4326)

Map.add_gdf(
    aoi,
    layer_name="AOI",
    style={"color": "red", "fillOpacity": 0}
)

# Zoom to AOI
xmin, ymin, xmax, ymax = aoi.total_bounds
Map.fit_bounds([[ymin, xmin], [ymax, xmax]])

# Display map
#Map

### Landsat True Color

In [63]:
#Landsat
# Convert GeoPandas -> Earth Engine FeatureCollection
aoi_ee = geemap.gdf_to_ee(aoi)

#Cloud masking
def cloudMask(image):
  cloudShadowBitmask = (1 << 3)
  cloudBitmask = (1 << 5)
  qa = image.select('QA_PIXEL')
  mask = qa.bitwiseAnd(cloudShadowBitmask).eq(0) \
                .And(qa.bitwiseAnd(cloudBitmask).eq(0))
  return image.updateMask(mask)

#Import Landsat image collection
CollectionLS = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
              .filterBounds(aoi_ee.geometry()) \
              .filterDate(startDate, endDate) \
              .filter(ee.Filter.calendarRange(1, 12, 'month')) \
              .map(cloudMask)

bands = ['SR_B4', 'SR_B3', 'SR_B2']

visualizationLS = {
  'bands': bands,
  'min': 7000,
  'max': 14000,
}

#Visualize RGB median
MedianLS = CollectionLS.median().clip(aoi_ee.geometry())
Map.addLayer(MedianLS, visualizationLS, 'Landsat | True Color')

### NDVI reference

In [64]:
def cloudMask(image):
  cloudShadowBitmask = (1 << 3)
  cloudBitmask = (1 << 5)
  qa = image.select('QA_PIXEL')
  mask = qa.bitwiseAnd(cloudShadowBitmask).eq(0) \
                .And(qa.bitwiseAnd(cloudBitmask).eq(0))
  return image.updateMask(mask)

#Import Landsat image collection
CollectionLS_ref = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
              .filterBounds(aoi_ee.geometry()) \
              .filterDate('2014-01-01', '2016-01-01') \
              .filter(ee.Filter.calendarRange(1, 12, 'month')) \
              .map(cloudMask)

MedianLS_ref = CollectionLS_ref.median().clip(aoi_ee.geometry())

#Normalized Difference Vegetation Index
ndvi_ref = MedianLS_ref.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')

# Compute min and max values over the AOI
ndvi_stats = ndvi_ref.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=aoi_ee.geometry(),
    scale=30,
    maxPixels=1e9
)

# Visualization parameters
vis_params = {
    'min': ndvi_stats.get('NDVI_min').getInfo(),
    'max': ndvi_stats.get('NDVI_max').getInfo(),
    'palette': ['red', 'white', 'green']
}

#Visualize
Map.addLayer(ndvi_ref, vis_params, 'Landsat | NDVI Reference period')

In [70]:
# Export to Drive
geemap.ee_export_image_to_drive(
  image=ndvi_ref, 
  description='NDVI_ref_'+site_name+'_2014_Layer',
  fileNamePrefix='NDVI_ref_'+site_name+'_2014',
  folder=site_name,
  region=aoi_ee.geometry(),
  scale=30,
  crs='EPSG:6875',
  maxPixels=1e10
)

### NDVI Current

In [66]:
#Normalized Difference Vegetation Index
ndvi = MedianLS.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')

# Compute min and max values over the AOI
ndvi_stats = ndvi.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=aoi_ee.geometry(),
    scale=30,
    maxPixels=1e9
)

# Visualization parameters
vis_params = {
    'min': ndvi_stats.get('NDVI_min').getInfo(),
    'max': ndvi_stats.get('NDVI_max').getInfo(),
    'palette': ['red', 'white', 'green']
}

#Visualize
Map.addLayer(ndvi, vis_params, 'Landsat | NDVI Current')

In [67]:
# Export to Drive
geemap.ee_export_image_to_drive(
  image=ndvi, 
  description='NDVI_current_'+site_name+'_'+year+'_Layer',
  fileNamePrefix='NDVI_current_'+site_name+'_'+year,
  folder=site_name,  
  region=aoi_ee.geometry(),
  scale=30,
  crs='EPSG:6875',
  maxPixels=1e10
)

### Canopy Cover

In [68]:
# Load Hansen Global Forest Change dataset
gfc = ee.Image("UMD/hansen/global_forest_change_2025_v1_13")

# Select tree cover percentage band
tree_cover = gfc.select("treecover2000").clip(aoi_ee.geometry())

# Visualization parameters
vis_params = {
    "min": 0,
    "max": 100,
    "palette": [
        "ffffff",  # 0%
        "ffffcc",
        "c2e699",
        "78c679",
        "31a354",
        "006837"   # 100%
    ]
}

# Add tree cover layer
Map.addLayer(tree_cover, vis_params, "Tree Cover (%)")

In [72]:
# Export to Drive
geemap.ee_export_image_to_drive(
  image=tree_cover, 
  description='Tree_Cover_'+site_name+'_2000_Layer',
  fileNamePrefix='Tree_Cover_'+site_name+'_2000',
  folder=site_name,
  region=aoi_ee.geometry(),
  scale=30,
  crs='EPSG:6875',
  maxPixels=1e10
)